# 05 — Export Final Pipeline for Serving
Bundle the fitted `StandardScaler`, the final `KMeans` model, and the
persona label mapping into a single joblib artifact that the FastAPI
app and Streamlit dashboard will both load.

In [1]:
import sys
sys.path.append("..")

import joblib
import pandas as pd

from src.preprocessing import FEATURE_COLUMNS

## Load fitted components from previous notebooks

In [2]:
scaler = joblib.load("../models/scaler.joblib")
kmeans_model = joblib.load("../models/kmeans_model.joblib")
persona_labels = joblib.load("../models/persona_labels.joblib")

print("Scaler features:", FEATURE_COLUMNS)
print("KMeans clusters:", kmeans_model.n_clusters)
print("Personas:", persona_labels)

Scaler features: ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
KMeans clusters: 7
Personas: {0: 'High-Income, High-Spender (Target)', 1: 'High-Income, High-Spender (Target)', 2: 'Low-Income, Low-Spender (Careful)', 3: 'Low-Income, High-Spender (Impulsive)', 4: 'High-Income, Low-Spender (Saver)', 5: 'Low-Income, High-Spender (Impulsive)', 6: 'High-Income, Low-Spender (Saver)'}


## Bundle into a single pipeline artifact
A dict is simplest and most transparent for this use case — a
full sklearn `Pipeline` object works too, but since persona labels
aren't a sklearn transformer, a dict bundle is more explicit.

In [3]:
pipeline_bundle = {
    "scaler": scaler,
    "kmeans_model": kmeans_model,
    "persona_labels": persona_labels,
    "feature_columns": FEATURE_COLUMNS,
}

joblib.dump(pipeline_bundle, "../models/clustering_pipeline.joblib")
print("Saved models/clustering_pipeline.joblib")

Saved models/clustering_pipeline.joblib


## Sanity check: reload and predict on a new customer
This mirrors exactly what `app/model_utils.py` will do.

In [4]:
bundle = joblib.load("../models/clustering_pipeline.joblib")

new_customer = pd.DataFrame([[28, 75, 82]], columns=bundle["feature_columns"])
scaled = bundle["scaler"].transform(new_customer)
cluster_id = int(bundle["kmeans_model"].predict(scaled)[0])
persona = bundle["persona_labels"][cluster_id]

print(f"Predicted cluster: {cluster_id}")
print(f"Persona: {persona}")

Predicted cluster: 0
Persona: High-Income, High-Spender (Target)


## Notes
- `models/clustering_pipeline.joblib` is now the single artifact the
  FastAPI app (`app/model_utils.py`) and the Streamlit dashboard both
  load — everything downstream depends only on this file plus
  `models/dbscan_model.joblib` (used only for the anomaly-detection
  dashboard page, not for live single-customer prediction, since
  DBSCAN can't natively predict on brand-new points).
- If you retrain later (new data, different K), just re-run notebooks
  02 → 05 and this file gets overwritten — nothing else needs to change.